# **Load Gemma 4 12B IT LLM**

In [1]:
# 1. Install packages and force TorchAudio to match PyTorch's CUDA 13.0 build
!pip install vllm openai nest_asyncio requests hf_transfer huggingface_hub --upgrade
!pip install --upgrade torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130 --force-reinstall

# 2. Authenticate and capture your token for the subprocess
from huggingface_hub import notebook_login, get_token
import os
import subprocess
import time
import requests
import nest_asyncio
from openai import OpenAI

notebook_login()

hf_token = get_token()
if not hf_token:
    raise ValueError("⚠️ No Hugging Face token found! Please make sure notebook_login() was successful.")

# 3. Configure high-speed transfers and vLLM internal timeouts
env = os.environ.copy()
env["HF_TOKEN"] = hf_token
env["HF_XET_HIGH_PERFORMANCE"] = "1"          # Updated high-performance transfer flag
env["VLLM_ENGINE_ITERATION_TIMEOUT_S"] = "1800" # Prevents vLLM from timing out on large models
env["VLLM_WORKER_TIMEOUT_S"] = "1800"         # Extends worker handshake windows

nest_asyncio.apply()

print("⏳ Launching google/gemma-4-12B-it on vLLM server with FP8 quantization...")

# Fixed syntax: Passed limit-mm-per-prompt as a proper JSON string
vllm_command = [
    "vllm", "serve", "google/gemma-4-12B-it",
    "--port", "8000",
    "--tensor-parallel-size", "1",
    "--quantization", "fp8",                     # Compresses weights on the fly for 24GB GPUs
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.90",
    "--limit-mm-per-prompt", '{"image": 0, "audio": 0}'  # Correct JSON format for vLLM CLI
]

# CRITICAL: We pipe stdout/stderr to a log file so you can debug if it fails!
log_file = open("vllm_server.log", "w")

vllm_process = subprocess.Popen(
    vllm_command,
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# Wait until the server is responsive
max_attempts = 90  # Extended to 15 minutes max for downloading & initializing
attempt = 0
server_ready = False

while attempt < max_attempts:
    # Check if the process died early (e.g., due to OOM or bad arguments)
    if vllm_process.poll() is not None:
        print("\n❌ vLLM process terminated unexpectedly! Printing last 20 lines of server logs:")
        with open("vllm_server.log", "r") as f:
            lines = f.readlines()
            print("".join(lines[-20:]))
        raise RuntimeError("vLLM Server crashed during initialization.")

    try:
        response = requests.get("http://localhost:8000/v1/models")
        if response.status_code == 200:
            print("\n✅ vLLM Server is successfully running and ready!")
            server_ready = True
            break
    except requests.exceptions.ConnectionError:
        pass

    attempt += 1
    time.sleep(10)
    print(f"⏳ Downloading weights & booting engine... (Attempt {attempt}/{max_attempts})")

if not server_ready:
    vllm_process.terminate()
    raise RuntimeError("❌ vLLM server timed out. Check vllm_server.log for details.")

# 4. Test query
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

response = client.chat.completions.create(
    model="google/gemma-4-12B-it",
    messages=[
        {"role": "system", "content": "You are an expert telecommunications network engineer."},
        {"role": "user", "content": "Explain how BGP route reflection works in large networks."}
    ],
    temperature=0.01,
    max_tokens=500
)

print("\n--- Model Response ---")
print(response.choices[0].message.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.4 MB/s eta 0:00:00
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.11.0.dev20260827+cu130
    Uninstalling torchaudio-2.11.0.dev20260827+cu130:
      Successfully uninstalled torchaudio-2.11.0.dev20260827+cu130
Looking in indexes: https://download.pytorch.org/whl/nightly/cu130
  Using cached https://download-r2.pytorch.org/whl/nightly/cu130/torchaudio-2.11.0.dev20260827%2Bcu130-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (7.4 kB)
Using cached https://download-r2.pytorch.org/whl/nightly/cu130/torchaudio-2.11.0.dev20260827%2Bcu130-cp313-cp313-manylinux_2_28_x86_64.whl (1.7 MB)
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.11.0
    Uninstalling torchaudio-2.11.0:
      Successfully uninstalled torchaudio-2.11.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following 

⏳ Launching google/gemma-4-12B-it on vLLM server with FP8 quantization...
⏳ Downloading weights & booting engine... (Attempt 1/90)
⏳ Downloading weights & booting engine... (Attempt 2/90)
⏳ Downloading weights & booting engine... (Attempt 3/90)
⏳ Downloading weights & booting engine... (Attempt 4/90)
⏳ Downloading weights & booting engine... (Attempt 5/90)
⏳ Downloading weights & booting engine... (Attempt 6/90)
⏳ Downloading weights & booting engine... (Attempt 7/90)
⏳ Downloading weights & booting engine... (Attempt 8/90)
⏳ Downloading weights & booting engine... (Attempt 9/90)
⏳ Downloading weights & booting engine... (Attempt 10/90)
⏳ Downloading weights & booting engine... (Attempt 11/90)
⏳ Downloading weights & booting engine... (Attempt 12/90)
⏳ Downloading weights & booting engine... (Attempt 13/90)
⏳ Downloading weights & booting engine... (Attempt 14/90)
⏳ Downloading weights & booting engine... (Attempt 15/90)
⏳ Downloading weights & booting engine... (Attempt 16/90)
⏳ Downl

In [2]:
from openai import OpenAI

# Connect to your already-running background vLLM server
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

response = client.chat.completions.create(
    model="google/gemma-4-12B-it",
    messages=[
        {"role": "system", "content": "You are an expert telecommunications network engineer."},
        {"role": "user", "content": "What 5G brings better than 4G."}
    ],
    temperature=0.01,
    max_tokens=500
)

print("\n--- Model Response ---")
print(response.choices[0].message.content)


--- Model Response ---
As a telecommunications network engineer, I view the transition from 4G (LTE) to 5G not just as a "faster internet" upgrade, but as a fundamental architectural shift in how data is transported, processed, and consumed.

While 4G was designed primarily for the **Mobile Broadband** era (smartphones and apps), 5G was engineered for the **Internet of Everything** era.

Here are the five core technical pillars that 5G brings over 4G:

### 1. Enhanced Mobile Broadband (eMBB) – The "Speed" Factor
This is what most consumers notice. 5G utilizes a much wider range of spectrum, including **mmWave (millimeter wave)** and **C-Band** frequencies.
*   **Throughput:** While 4G tops out at roughly 100–150 Mbps in real-world conditions, 5G can theoretically reach speeds of 10–20 Gbps.
*   **Capacity:** 5G can handle a much higher density of devices. In a crowded stadium or city center, 4G often "chokes" because the cell site reaches its maximum connection limit. 5G uses **Massiv

# **1. Environment Initialisation**

## **1.1. Load Dependencies**

### **Install All Required Libraries**

In [ ]:
# Install vLLM and tools to allow background web tasks inside a notebook
!pip install vllm nest_asyncio openai requests

In [ ]:
# =============================================================================
# Retriever Implementation
# INSTALL DEPENDENCIES
# =============================================================================

# Install the libraries required for telecom RAG data acquisition,
# document processing, embeddings and vector retrieval.
#
# The current runtime is CPU-based because inference is not yet being
# performed. GPU-specific acceleration can be enabled later when required.

!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    safetensors \
    sentencepiece \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-docx \
    python-pptx \
    beautifulsoup4 \
    pyarrow \
    tqdm

print("All required Module 2 libraries installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 19.7 MB/s eta 0:00:00
All required Module 2 libraries installed successfully.


### **Import All Required Library**

In [ ]:
# =============================================================================
# Retriever Implementation
# IMPORT REQUIRED LIBRARIES
# =============================================================================

# ---------------------------------------------------------------------------
# Core scientific stack
# ---------------------------------------------------------------------------
import numpy as np
import scipy
import pandas as pd

# ---------------------------------------------------------------------------
# Standard Python libraries
# ---------------------------------------------------------------------------
import os
import gc
import json
import shutil
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# Progress monitoring
# ---------------------------------------------------------------------------
from tqdm.auto import tqdm

# Data / document processing
# ---------------------------------------------------------------------------
import pyarrow
import pyarrow.parquet as pq
from docx import Document
from bs4 import BeautifulSoup
from pypdf import PdfReader

# ---------------------------------------------------------------------------
# HTTP / source acquisition
# ---------------------------------------------------------------------------
import requests

# ---------------------------------------------------------------------------
# PyTorch
# ---------------------------------------------------------------------------
import torch

# ---------------------------------------------------------------------------
# Hugging Face / LLM inference
# ---------------------------------------------------------------------------
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

from huggingface_hub import login, HfApi, snapshot_download

# ---------------------------------------------------------------------------
# Embeddings
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# Vector similarity search
# ---------------------------------------------------------------------------
import faiss

# ---------------------------------------------------------------------------
# PDF document processing
# ---------------------------------------------------------------------------
from pypdf import PdfReader

print("All required libraries imported successfully.")

# Display key package versions for reproducibility
print("\nPackage Versions")
print("-" * 40)
print(f"NumPy           : {np.__version__}")
print(f"SciPy           : {scipy.__version__}")
print(f"Pandas          : {pd.__version__}")
print(f"PyTorch         : {torch.__version__}")
print(f"FAISS           : {faiss.__version__}")
print(f"PyArrow         : {pyarrow.__version__}")

All required libraries imported successfully.

Package Versions
----------------------------------------
NumPy           : 2.1.3
SciPy           : 1.16.3
Pandas          : 2.2.3
PyTorch         : 2.11.0+cu128
FAISS           : 1.15.0
PyArrow         : 18.1.0


### **Import Datasets from Kaggle**

In [2]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

cliffordimaguezegie_telecom_benchmark_path = kagglehub.dataset_download('cliffordimaguezegie/benchmark')

print('Data source import complete.')


100%|██████████| 21.9k/21.9k [00:00<00:00, 38.7MB/s]

Extracting files...
Data source import complete.


In [4]:
print("QUESTIONS:")
print(cliffordimaguezegie_telecom_benchmark_path)

QUESTIONS:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1


In [5]:
from pathlib import Path


BENCHMARK_DIR = Path(
    cliffordimaguezegie_telecom_benchmark_path
)

print("BENCHMARK :", BENCHMARK_DIR)

BENCHMARK : /root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1


In [6]:
# =============================================================================
# RAG V1 — INSPECT BENCHMARK DATASET
# =============================================================================

print("=" * 90)
print("RAG V1 — BENCHMARK DATASET CONTENTS")
print("=" * 90)

for file_path in sorted(
    BENCHMARK_DIR.rglob("*")
):

    if file_path.is_file():

        print(
            file_path.relative_to(
                BENCHMARK_DIR
            )
        )

print("=" * 90)

RAG V1 — BENCHMARK DATASET CONTENTS
track1_20_questions.json
track2_final_32_questions.json


In [7]:
# =============================================================================
# LOAD BENCHMARK QUESTION BANKS
# =============================================================================

import json


TRACK1_FILE = (
    BENCHMARK_DIR
    / "track1_20_questions.json"
)

TRACK2_FILE = (
    BENCHMARK_DIR
    / "track2_final_32_questions.json"
)


# =============================================================================
# LOAD TRACK 1
# =============================================================================

with open(
    TRACK1_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions = json.load(
        file
    )


# =============================================================================
# LOAD TRACK 2
# =============================================================================

with open(
    TRACK2_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions_track2 = json.load(
        file
    )


# =============================================================================
# VALIDATION
# =============================================================================

print("=" * 90)
print("RAG V1 — BENCHMARK QUESTION BANKS LOADED")
print("=" * 90)

print("\nTRACK 1")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions[0]['id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions[-1]['id']}"
)


print("\nTRACK 2")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions_track2)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions_track2[0]['evaluation_id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions_track2[-1]['evaluation_id']}"
)


# =============================================================================
# COUNT CHECKS
# =============================================================================

if len(benchmark_questions) != 20:

    raise RuntimeError(
        f"Track 1 expected 20 questions, "
        f"found {len(benchmark_questions)}."
    )


if len(benchmark_questions_track2) != 32:

    raise RuntimeError(
        f"Track 2 expected 32 questions, "
        f"found {len(benchmark_questions_track2)}."
    )


print("\n" + "=" * 90)
print("TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED")
print("=" * 90)

RAG V1 — BENCHMARK QUESTION BANKS LOADED

TRACK 1
------------------------------------------------------------
Questions : 20
First ID  : Q01
Last ID   : Q20

TRACK 2
------------------------------------------------------------
Questions : 32
First ID  : T2-01
Last ID   : T2-32

TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED


# **2. Inference Pipeline**

## **2.2. Test Response**

In [12]:
from openai import OpenAI

# Connect to your already-running background vLLM server
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

response = client.chat.completions.create(
    model="google/gemma-4-12B-it",
    messages=[
        {"role": "system", "content": "You are an expert telecommunications network engineer."},
        {"role": "user", "content": "What is 5G."}
    ],
    temperature=0.01,
    max_tokens=500
)

print("\n--- Model Response ---")
print(response.choices[0].message.content)


--- Model Response ---
To understand 5G, it is best to view it not just as "faster 4G," but as a fundamental architectural shift in how telecommunications networks are built and how data is transported.

As a network engineer, I break down 5G into four core pillars: **Spectrum, Architecture, Latency, and Use Cases.**

---

### 1. The Spectrum (The "Highway" Width)
The primary difference between 4G and 5G is the utilization of different radio frequencies. 5G operates across three distinct bands:

*   **Low-Band (Sub-1 GHz):** Similar to 4G. It travels long distances and penetrates walls easily but offers speeds only slightly faster than current LTE.
*   **Mid-Band (1 GHz – 6 GHz):** This is the "sweet spot." It provides a balance of significant capacity and decent coverage. This is where most 5G deployments currently live.
*   **High-Band (mmWave - 24 GHz to 100 GHz):** These are extremely high frequencies. They offer massive "pipes" of data (multi-gigabit speeds) but have very short r

# **3. Telecom Benchmark Evaluation**

## **General LLM Inference**

### **Track 1 Inference**

In [8]:
import json
import time
from openai import OpenAI

# 1. Connect to your local vLLM server running on port 8000
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

# 2. Use your loaded benchmark questions variable
questions_template = benchmark_questions

results = []

print(f"🚀 Starting rapid inference on {len(questions_template)} questions via vLLM...")
start_time = time.time()

# 3. Loop through questions and query the model securely
for item in questions_template:
    # Safely handle different possible key names for IDs and categories
    q_id = item.get("id") or item.get("question_id") or item.get("index") or "Unknown"
    category = item.get("category") or item.get("topic") or "General"

    # Automatically detect whether your dictionary uses 'prompt', 'question', 'text', or 'query'
    prompt_text = (
        item.get("prompt") or
        item.get("question") or
        item.get("text") or
        item.get("query")
    )

    if not prompt_text:
        print(f"⚠️ Skipping Question ID {q_id}: Could not find question text in keys ({list(item.keys())})")
        continue

    print(f"Processing Question ID {q_id}...")

    try:
        response = client.chat.completions.create(
            model="google/gemma-4-12B-it",
            messages=[
                {"role": "system", "content": "You are an expert telecommunications network engineer."},
                {"role": "user", "content": str(prompt_text)}
            ],
            temperature=0.01,
            max_tokens=1024
        )

        answer = response.choices[0].message.content

        # Store successful result
        results.append({
            "id": q_id,
            "category": category,
            "prompt": prompt_text,
            "response": answer,
            "status": "success"
        })

    except Exception as e:
        print(f"❌ Error on Question {q_id}: {e}")
        # Store error state so your dataset doesn't break
        results.append({
            "id": q_id,
            "category": category,
            "prompt": prompt_text,
            "response": str(e),
            "status": "error"
        })

total_elapsed = time.time() - start_time
print(f"\n✅ Completed benchmark batch in {total_elapsed:.2f} seconds!")

# 4. Save structured outputs to a JSON file
output_filename = "gemma4_telecom_benchmark_results.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"📁 Results successfully saved to local file: '{output_filename}'")

🚀 Starting rapid inference on 20 questions via vLLM...
Processing Question ID Q01...
Processing Question ID Q02...
Processing Question ID Q03...
Processing Question ID Q04...
Processing Question ID Q05...
Processing Question ID Q06...
Processing Question ID Q07...
Processing Question ID Q08...
Processing Question ID Q09...
Processing Question ID Q10...
Processing Question ID Q11...
Processing Question ID Q12...
Processing Question ID Q13...
Processing Question ID Q14...
Processing Question ID Q15...
Processing Question ID Q16...
Processing Question ID Q17...
Processing Question ID Q18...
Processing Question ID Q19...
Processing Question ID Q20...

✅ Completed benchmark batch in 1221.91 seconds!
📁 Results successfully saved to local file: 'gemma4_telecom_benchmark_results.json'


### **Track 2 Inference**

In [9]:
import json
import time
from openai import OpenAI

# 1. Connect to your local vLLM server running on port 8000
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

# 2. Use your loaded Track 2 benchmark questions variable
questions_template = benchmark_questions_track2

results = []

print(f"🚀 Starting rapid inference on {len(questions_template)} questions via vLLM...")
start_time = time.time()

# Helper function to find keys case-insensitively
def find_key(dictionary, possible_keys):
    for pk in possible_keys:
        for k in dictionary.keys():
            if k.lower() == pk.lower():
                return dictionary[k]
    return None

# 3. Loop through questions and query the model securely
for idx, item in enumerate(questions_template, start=1):
    # Safely search for ID across various common casings/names, fallback to sequential T2-XX format
    q_id = (
        find_key(item, ["id", "question_id", "qid", "index", "number"]) or
        f"T2-{idx:02d}"
    )

    category = find_key(item, ["category", "topic", "domain"]) or "General"

    # Safely search for prompt/question text
    prompt_text = find_key(item, ["prompt", "question", "text", "query", "content"])

    if not prompt_text:
        print(f"⚠️ Skipping Track 2 Index {idx}: Could not find question text in keys ({list(item.keys())})")
        continue

    print(f"Processing Question ID {q_id}...")

    try:
        response = client.chat.completions.create(
            model="google/gemma-4-12B-it",
            messages=[
                {"role": "system", "content": "You are an expert telecommunications network engineer."},
                {"role": "user", "content": str(prompt_text)}
            ],
            temperature=0.01,
            max_tokens=1024
        )

        answer = response.choices[0].message.content

        # Store successful result
        results.append({
            "id": q_id,
            "category": category,
            "prompt": prompt_text,
            "response": answer,
            "status": "success"
        })

    except Exception as e:
        print(f"❌ Error on Question {q_id}: {e}")
        # Store error state so your dataset doesn't break
        results.append({
            "id": q_id,
            "category": category,
            "prompt": prompt_text,
            "response": str(e),
            "status": "error"
        })

total_elapsed = time.time() - start_time
print(f"\n✅ Completed Track 2 benchmark batch in {total_elapsed:.2f} seconds!")

# 4. Save structured outputs to a JSON file specifically for Track 2
output_filename = "gemma4_track2_benchmark_results.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"📁 Results successfully saved to local file: '{output_filename}'")

🚀 Starting rapid inference on 32 questions via vLLM...
Processing Question ID T2-01...
Processing Question ID T2-02...
Processing Question ID T2-03...
Processing Question ID T2-04...
Processing Question ID T2-05...
Processing Question ID T2-06...
Processing Question ID T2-07...
Processing Question ID T2-08...
Processing Question ID T2-09...
Processing Question ID T2-10...
Processing Question ID T2-11...
Processing Question ID T2-12...
Processing Question ID T2-13...
Processing Question ID T2-14...
Processing Question ID T2-15...
Processing Question ID T2-16...
Processing Question ID T2-17...
Processing Question ID T2-18...
Processing Question ID T2-19...
Processing Question ID T2-20...
Processing Question ID T2-21...
Processing Question ID T2-22...
Processing Question ID T2-23...
Processing Question ID T2-24...
Processing Question ID T2-25...
Processing Question ID T2-26...
Processing Question ID T2-27...
Processing Question ID T2-28...
Processing Question ID T2-29...
Processing Questi